# Imports

In [2]:
## conda env: stereo_visionn
import os
import cv2
import glob
import numpy as np
from rtmlib import draw_skeleton, draw_bbox
from Util.util import *
import time
import json
from IPython.display import display, clear_output

import pyzed.sl as sl
import matplotlib.pyplot as plt

with open('./Util/models.json') as f:
    models = json.load(f)

with open("./Util/properties.json", "r") as json_file:
    properties = json.load(json_file)

%matplotlib ipympl
# %matplotlib notebook
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

# 2D Keypoints

## Model setup

In [2]:
device = "cuda"  # cpu, cuda, mps
backend = "onnxruntime"  # opencv, onnxruntime, openvino
detector_name = 'YOLOX_nano' # 'YOLOX_l_COCO','YOLOX_nano','YOLOX_tiny','YOLOX_s','YOLOX_m','YOLOX_l','YOLOX_x'
pose_name = 'RTMPose_x' # (26) 'RTMPose_t', 'RTMPose_s', 'RTMPose_m', 'RTMPose_l', 'RTMPose_m2', 'RTMPose_l2', 'RTMPose_x', (133) 'RTMW_l', 'RTMW_x'
kpt_labels = models['pose_models']["26"]["kpt_labels"]

custom = Custom(det_class='YOLOX',#'RTMDet',
                det=models['detectors'][detector_name]['path'],
                det_input_size=models['detectors'][detector_name]['input_size'],
                pose_class='RTMPose',
                pose=models['pose_models']["26"]["models"][pose_name]['path'],
                pose_input_size=models['pose_models']["26"]["models"][pose_name]['input_size'],
                backend=backend,
                device=device) 

load C:\Users\unger\.cache\rtmlib\hub\checkpoints\yolox_nano_8xb8-300e_humanart-40f6f0d0.onnx with onnxruntime backend
load C:\Users\unger\.cache\rtmlib\hub\checkpoints\rtmpose-x_simcc-body7_pt-body7-halpe26_700e-384x288-7fb6e239_20230606.onnx with onnxruntime backend


## 2D inference

In [3]:
input_folder = ".\\stereo_videos\\validation_test"

for g in glob.glob(os.path.join(input_folder, "*.avi")):
    print(f"processing: {g}")

    keypoints_over_time = []
    
    cap = cv2.VideoCapture(g)
    if cap.isOpened() == False:
        print("Error opening video file")

    total_num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    processed_num_frames = 0
    start_time = time.time_ns()

    # Read until video is completed
    while cap.isOpened():

        # Capture frame-by-frame
        ret, frame = cap.read()

        if ret == True and frame is not None:
            processed_num_frames += 1
            width = frame.shape[1]
            height = frame.shape[0]

            # inference
            keypoints, scores = custom(frame)

            # store results (frames * kpt * (x,y,confidence))
            current_values = np.squeeze(np.dstack((keypoints, scores)))

            # if multiple ppl detected, only keep the first one
            if current_values.shape != (26, 3):
                current_values = current_values[0, :, :]
            keypoints_over_time.append(current_values)

            # img_show = draw_skeleton(frame, keypoints, scores, kpt_thr=0.2)
            # boxes = [pose_to_bbox(x) for x in keypoints]
            # img_show = draw_bbox(frame, boxes, (0, 0, 255))

            ## check to see in ankle tracking switches when changing direction
            # cv2.circle(img_show,(round(keypoints[0][16][0]),round(keypoints[0][16][1])),5,(0,0,255),3 )

            # cv2.imshow("Image", cv2.resize(img_show, (int(width / 2), int(height / 2))))

            clear_output(wait=True)
            end_time = time.time_ns()
            elapsed_time = (end_time - start_time) / 1000000000

            if int(elapsed_time % 10) == 0:

                fps = round(processed_num_frames / elapsed_time, 2)
                done_ratio = processed_num_frames / total_num_frames
                expected_duration_min = elapsed_time / (done_ratio * 60)

                display(
                    f"exp_duration: {round(expected_duration_min,2)} minutes, elapsed: {round(elapsed_time/60,2)} minutes"
                )
                display(f"done: {round(done_ratio*100,2)}%, fps: {fps}")

            # Press Q on keyboard to exit
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

        else:
            break

    cap.release()
    cv2.destroyAllWindows()

    keypoints_over_time = np.asarray(keypoints_over_time[0:])
    print(f"shape of coordinates: {keypoints_over_time.shape}")
    

shape of coordinates: (3908, 26, 3)


## Create new keypoint (mid_heel)

In [ ]:
left_heel = keypoints_over_time[:,24,:]
right_heel = keypoints_over_time[:,25,:]

# new keypoints between ankles 
# note: confidence is also the avg of ankles
mid_heel = (left_heel + right_heel)/2

# add the new keypoint to the rest
keypoints_over_time = np.concat((keypoints_over_time, np.expand_dims(mid_heel, axis=1)), axis=1)

kpt_labels = np.append(kpt_labels, "mid_heel")
print(f'keypoints shape (w new one added): {keypoints_over_time.shape}')

## Filter data 
(so it's less jiggly for depth)

In [4]:
filtered_2d_keypoints = []

for keypoint_id, keypoint in enumerate(keypoints_over_time.transpose(1, 0, 2)):
    result = filter_2d_keypoint(keypoint,6,7, 60, False,f'{keypoint_id}')
    filtered_2d_keypoints.append(result)

filtered_2d_keypoints = np.array(filtered_2d_keypoints).transpose(1, 0, 2)
print("Filtered data shape:", filtered_2d_keypoints.shape)

Filtered data shape: (3908, 26, 3)


## Calculate scale

In [ ]:
# participant height in meters
HEIGHT_METERS = 1.8

# y coords of the top_head and mid_heel keypoints
top_head_y = keypoints_over_time[:,17,1]
mid_heel_y = mid_heel[:,1]

# participant height in pixels
height_px = (mid_heel_y - top_head_y)

# how many meters one pixel is in each frame
scale = HEIGHT_METERS/height_px

print(f"scale shape: {scale.shape}")
# scale[:5]

## Save data

In [5]:
out_file_name = os.path.join(input_folder, f"{os.path.basename(g).split('.')[0]}.npz")

with open(out_file_name, "w") as f:
    np.savez(
        out_file_name,
        raw_keypoints=keypoints_over_time,
        keypoints_2d=filtered_2d_keypoints,
        kpt_labels=kpt_labels,
        # scale=scale,
    )

print(f"Saved 2d keypoint to {out_file_name}")

Saved 2d keypoint to .\stereo_videos\validation_test\43916681.npz


In [ ]:
kpt = 0
filter_2d_keypoint(keypoints_over_time[:,kpt,:],6,15,60,True, f"{kpt}")

In [ ]:
# bandpass
from scipy.signal import butter, filtfilt

x, y, d = keypoints_over_time[:, 0, :].T

lowcut = 1  # cut fr for lowpass
# highpass = 5 # cut fr for highpass
bandpass = [0.5, 5]
b, a = butter(2, bandpass, fs=60, btype="band", analog=False)
fx = filtfilt(b, a, x)
fy = filtfilt(b, a, y)

plt.close('all')
plt.figure(figsize=(13,5))
plt.subplot(211)
plt.plot(x, 'b', alpha=0.5)
plt.plot(fx+np.min(x),'r', alpha=0.5)
plt.title('x')


plt.subplot(212)
plt.plot(y, 'b', alpha=0.5)
plt.plot(fy + np.min(y), 'r', alpha=0.5)
plt.title('y')

plt.tight_layout()
plt.legend()

## Visualization

In [ ]:
input_vid = ".\\stereo_videos\\validation_test\\43916681.avi"

vid_basename = os.path.basename(input_vid)
folder_name = os.path.dirname(input_vid)
npz_basename = f"{vid_basename.split(".")[0]}.npz"

load_path = os.path.join(folder_name, npz_basename)
loaded_data = np.load(load_path)
print(f"loaded data : {load_path}\nwith keys: {list(loaded_data.keys())}")

keypoints = loaded_data["keypoints_2d"]
# keypoints = loaded_data["raw_keypoints"]
kpt_labels = loaded_data["kpt_labels"]
scale = loaded_data["scale"]



cap = cv2.VideoCapture(input_vid)
if cap.isOpened() == False:
    print("Error opening video file")

total_num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
frame_idx = 0

# Read until video is completed
while cap.isOpened():

    # Capture frame-by-frame
    ret, frame = cap.read()

    if ret == True and frame is not None:
        frame_idx += 1
        width = frame.shape[1]
        height = frame.shape[0]


        for kpt, conf in zip(keypoints[frame_idx,:,:2],keypoints[frame_idx,:,2]):
            cv2.circle(frame,(int(kpt[0]),int(kpt[1])),radius=3, color=(0,0,255), thickness=2 )
            # cv2.putText(frame, f"{conf:.2f}",(int(kpt[0]),int(kpt[1])),cv2.FONT_HERSHEY_SIMPLEX,fontScale=0.75,color = (255, 255, 255))
            

        cv2.imshow("Image", cv2.resize(frame, (int(width / 2), int(height / 2))))
        time.sleep(0.01)

        # Press Q on keyboard to exit
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    else:
        break

cap.release()
cv2.destroyAllWindows()
    

# 3D Keypoints

## Load 2d npz data

In [3]:
npz_load_path = "stereo_videos\\validation_test\\43916681.npz"
loaded_data = np.load(npz_load_path)

svo_path = f"{os.path.splitext(npz_load_path)[0]}.svo2"

# raw_keypoints = loaded_data["raw_keypoints"]
keypoints_2d = loaded_data["keypoints_2d"]
kpt_labels = loaded_data["kpt_labels"]
# scale = loaded_data["scale"]

print(f"loaded data : {npz_load_path}\nwith keys: {list(loaded_data.keys())}")
print(f"2d keypoint shape: {keypoints_2d.shape}")#frames x kpt x (x,y,2d_confidence)


KeyError: 'keypoints_2d is not a file in the archive'

## Depth

In [4]:
# TODO: update ZED sdk, try it with latest models, methods

keypoints_3d = []
keypoints_xyz = []

# Create a ZED camera object
zed = sl.Camera()

input_type = sl.InputType()
input_type.set_from_svo_file(svo_path)  # Set init parameter to run from the .svo
init_parameters = sl.InitParameters(input_t=input_type, svo_real_time_mode=False)

init_parameters.depth_mode = sl.DEPTH_MODE.NEURAL_PLUS
init_parameters.coordinate_units = sl.UNIT.METER  # CENTIMETER, METER, MILLIMETER
# init_parameters.coordinate_system = sl.COORDINATE_SYSTEM.RIGHT_HANDED_Y_UP

# TODO: see how own filtering does without this built in one
init_parameters.depth_stabilization = 75 # 0 to trun it off, otherwise 1-100 linear. default is 30

# Open the ZED
err = zed.open(init_parameters)
framerate = zed.get_camera_information().camera_configuration.fps
resolution = zed.get_camera_information().camera_configuration.resolution

svo_depth = sl.Mat()
svo_image = sl.Mat()
# svo_confidance = sl.Mat()
# svo_xyz = sl.Mat()

frame_idx = 0

while frame_idx < keypoints_2d.shape[0] - 1 and zed.grab() == sl.ERROR_CODE.SUCCESS:

    current_keypoints = []
    # current_xyz = []

    # Get frame count
    frame_idx = zed.get_svo_position()

    # get color image and depth map
    zed.retrieve_measure(svo_depth, sl.MEASURE.DEPTH)
    zed.retrieve_image(svo_image, sl.VIEW.LEFT) # BGRA image
    # zed.retrieve_measure(svo_confidance, sl.MEASURE.CONFIDENCE)
    # zed.retrieve_measure(svo_xyz, sl.MEASURE.XYZRGBA) # trash documentation, no clue what this returns

    # img = svo_image.get_data()
    depth_map = svo_depth.get_data()
    depth_map = np.transpose(depth_map)
    # depth_confidance_map = svo_confidance.get_data()
    # point_cloud = svo_xyz.get_data()
    # print(f"point_cloud shape: {point_cloud.shape}")
    

    # get coords for all keypoints within the current frame
    # note: x,y coords are in pixels, depth is in meters
    for x, y, xy_conf in keypoints_2d[frame_idx, :, :]:

        depth = depth_map[int(x), int(y)]
        # b, g, r, a = img[int(x), int(y)]
        # depth_conf = depth_confidance_map[int(x), int(y)]
        # xyz_point = point_cloud[int(x), int(y)]

        # cv2.circle(img, (int(x), int(y)), 5, (0, 0, 100 * 255 / 100), 3)
        
        # scale x y coordinates now, they are not used as indices anymore
        # x = x * scale[frame_idx]
        # y = y * scale[frame_idx]

        current_keypoints.append(np.array([depth, x, y]))
        # current_xyz.append(xyz_point)
        

        
        # print(f"x: {x:.0f}, y: {y:.0f}, depth: {depth:.01f}, xy_conf: {xy_conf:.01f}, depth_conf: {depth_conf:.0f}")

    current_keypoints = np.array(current_keypoints)
    keypoints_3d.append(current_keypoints)
    # keypoints_xyz.append(current_xyz)

#     ## show frame
#     cv2.imshow("vid", cv2.resize(img, (800, 600)))

#     if cv2.waitKey(25) & 0xFF == ord("q"):
#         break


# cv2.destroyAllWindows()
zed.close()

# shape: frames * kpts * (x, y, depth,xy_conf, depth_conf)
keypoints_3d = np.array(keypoints_3d)
print(f"keypoints 3d shape: {keypoints_3d.shape}")

# keypoints_xyz = np.array(keypoints_xyz)
# print(f"keypoints_xyz shape: {keypoints_xyz.shape}")

keypoints 3d shape: (3908, 26, 3)


In [9]:
save_path = "./stereo_videos/validation_test/xyz.npz"
with open(save_path, "w") as f:
    np.savez(
        npz_load_path,
        keypoints_3d=keypoints_3d,
        kpt_labels=kpt_labels,
        R = R,
        t = tvec
    )

print(f"Saved 3D keypoints to {npz_load_path}")

Saved 3D keypoints to stereo_videos\validation_test\xyz.npz


## Save data

In [ ]:
raw_keypoints = keypoints_over_time

In [ ]:
with open(npz_load_path, "w") as f:
    np.savez(
        npz_load_path,
        raw_keypoints=raw_keypoints,
        keypoints_2d=keypoints_2d,
        keypoints_3d=keypoints_3d,
        keypoints_xyz=keypoints_xyz,
        kpt_labels=kpt_labels,
        scale=scale,
    )

print(f"Saved 3D keypoints to {npz_load_path}")

## Reshape data for qualisys

In [ ]:
npz_load_path = "stereo_videos\\validation_test\\43916681.npz"
loaded_data = np.load(npz_load_path)

keypoints_3d = loaded_data["keypoints_3d"]
kpt_labels = list(loaded_data["kpt_labels"])
print(f"loaded data : {npz_load_path}\nwith keys and shapes:")
[f"{key}:    {loaded_data[key].shape}" for key in list(loaded_data.keys())]

# depth = keypoints_3d[:,0,:]

## Filter depth

In [ ]:
from Util.util import interpolate_gaps, filter_data, step_detection

with open("./Util/properties.json", "r") as json_file:
        properties = json.load(json_file)

In [ ]:
keypoints_3d = np.transpose(keypoints_3d, (-2,-1,0))

gap_filled = interpolate_gaps(keypoints_3d,60,0.25)
filtered = filter_data(gap_filled,60,7,4,0.25)
filtered.shape

## Save Data

In [ ]:
with open(npz_load_path, "w") as f:
    np.savez(npz_load_path, keypoints_2d=keypoints_2d, keypoints_3d=keypoints_3d, filtered=filtered, kpt_labels=kpt_labels, scale=scale)

print(f"Saved 3D keypoints to {npz_load_path}")

# WP

## Loading data

In [8]:
# load qualisys data
file_path = "stereo_videos\\validation_test\\qualisys_mate_walking.npz"
data = np.load(file_path, allow_pickle=True)
pose_data = data['pose_data']


## load stereo data
npz_load_path = "stereo_videos\\validation_test\\43916681.npz"
loaded_data = np.load(npz_load_path)

# filtered = loaded_data["filtered"]
# raw_keypoints = loaded_data["raw_keypoints"]
keypoints_3d = loaded_data["keypoints_3d"]
# keypoints_2d = loaded_data["keypoints_2d"]
# keypoints_xyz = loaded_data["keypoints_xyz"]
# keypoints_3d_corrected = loaded_data["keypoints_3d_corrected"]

# scale = loaded_data["scale"]
kpt_labels = list(loaded_data["kpt_labels"])

print(f"loaded data : {npz_load_path}\nwith keys and shapes:")
[f"{key}:    {loaded_data[key].shape}" for key in list(loaded_data.keys())]



loaded data : stereo_videos\validation_test\43916681.npz
with keys and shapes:


['keypoints_3d:    (3908, 26, 3)',
 'kpt_labels:    (26,)',
 'R:    (3, 3)',
 't:    (3, 1)']

In [ ]:
# stereo labels
for idx,label in enumerate(kpt_labels):
    print(idx, label)

In [ ]:
# Qualisys labels
for i,l in enumerate(data['kpt_labels']):
    print(i,l)

In [ ]:
pose_data.shape, keypoints_3d.transpose((1,2,0)).shape

In [10]:
keypoints_3d[0,0]

array([  5.81535292, 983.66761812, 435.62088928])

## Qualisys - Stereo comparison

In [ ]:
keypoints_3d_corrected.shape, xyz_corrected.transpose(1,2,0).shape

In [ ]:
from scipy.signal import butter, filtfilt

bandpass = [0.3, 3]
b, a = butter(4, bandpass, fs=60, btype="band", analog=False)

keypoint_name = "right_ankle"
contra_keypoint_name = "left_ankle"

stereo_idx = kpt_labels.index(keypoint_name)
qualisys_idx = list(data["kpt_labels"]).index(keypoint_name)

pose_data = filter_data(pose_data, 100, 7, 4, 0.25)
fd3 = filter_data(keypoints_3d.transpose((1, 2, 0)), 60, 4, 6, 0.25)[stereo_idx, 0, :]

# Qualisys
Qd = pose_data[qualisys_idx, 0, :] / 1000
Qx = pose_data[qualisys_idx, 2, :] / 1000
Qy = pose_data[qualisys_idx, 1, :] / 1000


# stereo
# rx, ry, rc = raw_keypoints[:, stereo_idx, :].T
# x2, y2, c2 = keypoints_2d[:, stereo_idx, :].T
# d3, x3, y3 = keypoints_3d[:, stereo_idx, :3].T

# fx = filtfilt(b, a, rx)
# fy = filtfilt(b, a, ry)


# rsx = np.multiply(rx, scale)
# rsy = np.multiply(ry, scale)

# sfy = np.multiply(fy, scale)  # scaled filtered vertical ax
# fsy = filtfilt(b, a, np.multiply(ry, scale))
# fsx = filtfilt(b, a, np.multiply(rx, scale))

# corrected stereo
xyz_corrected = filter_data(keypoints_3d_corrected, 60, 2.5, 4,0.25)
xyz_corrected = xyz_corrected.transpose(2, 0, 1)

X = xyz_corrected[225:, stereo_idx, 0]
Y = -xyz_corrected[225:, stereo_idx, 1]
Z = -xyz_corrected[225:, stereo_idx, 2]

contra_Z = -xyz_corrected[225:, kpt_labels.index(contra_keypoint_name), 2]





# nans = np.isnan(Z)

# # if there are nans, interpolate the missing values for subsequent filtering
# if np.any(nans):
#     valid_indices = ~nans
#     Z[nans] = np.interp(np.flatnonzero(nans), np.flatnonzero(valid_indices), Z[valid_indices])


# bandpass = [0.7, 2.5]
# b, a = butter(1, bandpass, fs=60, btype="band", analog=False)
# Z = filtfilt(b,a,Z)

# nans = np.isnan(contra_Z)

# # if there are nans, interpolate the missing values for subsequent filtering
# if np.any(nans):
#     valid_indices = ~nans
#     contra_Z[nans] = np.interp(np.flatnonzero(nans), np.flatnonzero(valid_indices), contra_Z[valid_indices])


# bandpass = [0.7, 2.5]
# b, a = butter(1, bandpass, fs=60, btype="band", analog=False)
# contra_Z = filtfilt(b,a,contra_Z)


tQ = np.linspace(0,len(Qd)/100,len(Qd))
tS = np.linspace(0,len(X)/60,len(X))
xticks = np.array([0,11,16,18,23,25,29,31,36,38,43,45,50,52,57,58,63,65])-3.3

figsize = (13, 10)
plt.close("all")
plt.figure(figsize=figsize)
plt.suptitle(keypoint_name)


# plot x (horizontal plane ax1)
plt.subplot(711)
plt.plot(tQ[1:],np.diff(Qy),'m',label = 'Gold')
# plt.plot(tQ,Qy, "m", label="Gold")
plt.xticks(xticks)
plt.xlim([min(xticks), max(xticks)])
plt.title("X axis (horizontal)")
plt.legend(loc="upper left")
plt.grid()

plt.subplot(712)
plt.plot(tS[1:],np.diff(X),'m',label='Stereo')
# plt.plot(tS,X, "m", label="Stereo X")
plt.xticks(xticks)
plt.xlim([min(xticks), max(xticks)])
plt.legend(loc="upper left")
plt.grid()


# plot y (horizontal plane ax2)
plt.subplot(713)
plt.plot(tQ[1:],np.diff(Qd),label='Gold')
# plt.plot(tQ,Qd, label="Gold")
plt.xticks(xticks)
plt.xlim([min(xticks), max(xticks)])
plt.title("Y axis (horizontal)")
plt.legend(loc="upper left")
plt.grid()

plt.subplot(714)
plt.plot(tS[1:],np.diff(Y),label='Stereo')
# plt.plot(tS,Y, label="Stereo")
plt.xticks(xticks)
plt.xlim([min(xticks), max(xticks)])
plt.legend(loc="upper left")
plt.grid()


# plot z (vertical plane)
plt.subplot(715)
plt.plot(tQ[1:],np.diff(Qx),'r',label = 'Gold')
# plt.plot(tQ,Qx, "r", label="Gold")
plt.xticks(xticks)
plt.xlim([min(xticks), max(xticks)])
plt.title("Z axis (vertical)")
plt.legend(loc="upper left")
plt.grid()

plt.subplot(716)
plt.plot(tS[1:],np.diff(Z),'r',label='Stereo')
# plt.plot(tS,Z, "r", label="Stereo")
plt.xticks(xticks)
plt.xlim([min(xticks), max(xticks)])
plt.legend(loc="upper left")
plt.grid()

# plt.subplot(717)
# # plt.plot(np.diff(-sfy)[250:],'r', label="Stereo")
# plt.plot(tS,contra_Z, "r", label="Stereo")
# plt.xticks(xticks)
# plt.xlim([min(xticks), max(xticks)])
# plt.legend(loc="upper left")
# plt.grid()

plt.tight_layout()

In [ ]:
xyz_corrected = xyz_corrected.transpose(1,2,0)
xyz_corrected[:,1,:] = - xyz_corrected[:,1,:]
xyz_corrected[:,2,:] = - xyz_corrected[:,2,:]

In [ ]:
with open(npz_load_path, "w") as f:
    np.savez(
        npz_load_path,
        keypoints_2d=keypoints_2d,
        keypoints_3d=keypoints_3d,
        keypoints_3d_corrected = keypoints_3d_corrected,
        keypoints_3d_filtered = xyz_corrected,
        # filtered=filtered,
        kpt_labels=kpt_labels,
        scale=scale,
    )

print(f"Saved 3D keypoints to {npz_load_path}")

In [ ]:

tS = np.linspace(0,len(X)/60,len(X))
xticks = np.array([0,11,16,18,23,25,29,31,36,38,43,45,50,52,57,58,63,65])-3.3


plt.figure(figsize=(15,5))
plt.subplot(311)
plt.plot(tS,Z)
plt.xticks(xticks)
plt.xlim([min(xticks), max(xticks)])
plt.hlines(0,min(xticks),max(xticks),'r')
plt.grid()

plt.subplot(312)
plt.plot(tS,contra_Z)
plt.xticks(xticks)
plt.xlim([min(xticks), max(xticks)])
plt.hlines(0,min(xticks),max(xticks),'r')

plt.grid()


Ztest = copy.deepcopy(Z)

mask = Ztest<0
Ztest[mask] = np.interp(np.flatnonzero(mask), np.flatnonzero(~mask), Ztest[~mask])
plt.subplot(313)
plt.plot(tS,Ztest)
plt.xticks(xticks)
plt.xlim([min(xticks), max(xticks)])
plt.hlines(0,min(xticks),max(xticks),'r')

plt.grid()

In [ ]:
Ztest[Ztest<0] = Ztest[Ztest<0]+contra_Z[Ztest<0]

In [ ]:
zt = -xyz_corrected[225:, stereo_idx, 2]

nans = np.isnan(zt)

# if there are nans, interpolate the missing values for subsequent filtering
if np.any(nans):
    valid_indices = ~nans
    zt[nans] = np.interp(np.flatnonzero(nans), np.flatnonzero(valid_indices), zt[valid_indices])

plt.close('all')
plt.figure(figsize=(15,2))
x = fft(zt)
N = len(x)
n = np.arange(N)
freq = n / (N / 60)

PSD = x * np.conj(x) / N
PSD = np.array(PSD)
indices =  PSD < 1.5 # indices 
PSDClean = PSD*indices 

# indices[0] = False

fhat = indices * x
ffilt = np.fft.ifft(fhat)

# plt.plot(freq, PSD)
# plt.ylim([0,5])
# plt.xlim([0,15])

plt.subplot(211)
plt.plot(freq,PSD)
plt.xlim([0,5])
plt.ylim([0,5])


plt.subplot(212)
plt.plot(ffilt,'m')
plt.plot(zt+0.6,'b')

plt.show()

In [ ]:
indices[:10]

In [ ]:
plt.close('all')
plt.figure(figsize=(13,6))

bandpass = [0.2, 15]
# orders = [1, 4,7, 8]
# band_mins = [0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
band_maxs = [4,5,8,10,15,20]
values = band_maxs
for idx, value in enumerate(values,1):
    b, a = butter(6, [0.2, value], fs=60, btype="band", analog=False)
    result = filtfilt(b,a,ry)
    plt.subplot(len(values),1,idx)
    plt.plot(result, label=str(value))
    plt.hlines(0,0,len(result),'r')
    plt.legend()
    

In [ ]:
# keypoints_over_time.transpose(1,2,0).shape, 
keypoints_3d.transpose(1,2,0).shape

In [ ]:
bandpass = [0.2, 4]
b, a = butter(6, bandpass, fs=60, btype="band", analog=False)

test = []

for kpt_idx, kpt in enumerate(raw_keypoints.transpose(1,2,0)):
    # fx = filtfilt(b, a, kpt[0,:])
    # fy = filtfilt(b, a, kpt[1,:])
    sx = np.multiply(kpt[0,:], scale)
    # fy = kpt[1,:]
    fy = filtfilt(b,a,kpt[1,:])
    sfy = np.multiply(fy, scale)

    depth = keypoints_3d.transpose(1,2,0)[kpt_idx,0,:]
    # print(fx.shape, fy.shape, depth.shape)
    test.append(np.stack((depth,sx,sfy)))
test= np.array(test)
print(test.shape)
type(test)
    

In [ ]:
faszom = "stereo_videos\\validation_test\\test.npz"
with open(faszom, "w") as f:
    np.savez(faszom, raw_keypoints = raw_keypoints, keypoints_2d=keypoints_2d, keypoints_3d=keypoints_3d, kpt_labels=kpt_labels, scale=scale, test = test)

print(f"Saved 3D keypoints to {faszom}")

## distortion correction

In [ ]:
with open("./Util/val_test_calib.npz","r") as f:
    stuff = np.load(f)
    print(stuff.keys())

In [ ]:
with open("./Util/camera_params.json", "r") as f:
    intrinsic_params = json.load(f)
cameraMatrix = np.array(intrinsic_params['SN43916681']["left_sensor"]["camera_matrix"])



f_x = cameraMatrix[0,0]
f_y = cameraMatrix[1,1]
c_x = cameraMatrix[0,2]
c_y = cameraMatrix[1,2]



In [ ]:
keypoints_xyz.transpose(1,2,0).shape

In [ ]:
keypoints_3d_corrected = []

for kpt in keypoints_3d.transpose(1,2,0):
    # print(kpt.shape)
    Z, u, v  = kpt[:3,:]

    X = (( u - c_x) * Z) / (f_x)
    Y = (( v - c_y) * Z) / (f_y)
    keypoints_3d_corrected.append(np.array([X,Z,Y]))
keypoints_3d_corrected = np.array(keypoints_3d_corrected)


In [ ]:
with open(npz_load_path, "w") as f:
    np.savez(
        npz_load_path,
        raw_keypoints=raw_keypoints,
        keypoints_2d=keypoints_2d,
        keypoints_3d=keypoints_3d,
        keypoints_xyz=keypoints_xyz,
        keypoints_3d_corrected=keypoints_3d_corrected,
        kpt_labels=kpt_labels,
        scale=scale,
    )

print(f"Saved 3D keypoints to {npz_load_path}")

## Spatial calibration

In [3]:
from Util.util import Pattern, get_points
import glob

# input/output folders
# data_dir = "./chessboard_data"
data_dir = "./stereo_videos/validation_test"
output_dir = "./stereo_videos/validation_test"

with open("./Util/camera_params.json", "r") as f:
    intrinsic_params = json.load(f)

# describe real world calibration pattern parameteres
board_pattern = Pattern(6, 4, 0.150)


# serial numbers, file names and images
sns_fns_imgs = [
    (
        get_serial_number(file),
        file.split(".png")[0].split("\\")[1],
        cv2.imread(file, cv2.IMREAD_GRAYSCALE),
    )
    for file in glob.glob(os.path.join(data_dir, "*.png"))
]

extrinsics = []
camera_coords = []


for sn, fn, img in sns_fns_imgs:
    print(f"Processing {fn}...")
    img_points, obj_points = get_points(img, fn, output_dir, board_pattern)
    print(f"sn: {sn}")
    cameraMatrix = np.array(intrinsic_params[sn]["left_sensor"]["camera_matrix"])
    distCoeffs = np.array(intrinsic_params[sn]["left_sensor"]["distortion_coeff"])

    f_x = cameraMatrix[0, 0]
    f_y = cameraMatrix[1, 1]
    c_x = cameraMatrix[0, 2]
    c_y = cameraMatrix[1, 2]

    success, rvec, tvec = cv2.solvePnP(
        obj_points,
        img_points,
        cameraMatrix,
        distCoeffs,
        useExtrinsicGuess=False,
        flags=cv2.SOLVEPNP_ITERATIVE,
    )  # CV_P3P ,CV_EPNP
    # success, rvec, tvec, inliners = cv.solvePnPRansac(obj_points, img_points,cameraMatrix,distCoeffs)
    if success:
        print(f"{fn}... OK")
        # extrinsics.append((sn, fn, np.concatenate((rvec, tvec), 1).reshape(1, 6), rvec, tvec))
        R, jac = cv2.Rodrigues(rvec)

        Xw = -np.matrix(R).T * np.matrix(tvec)
        camera_coords.append(Xw)

        print(f"distance: {round(np.linalg.norm(Xw),2)} m")

    else:
        print(f"{fn}... FAILED")


Processing mate_walking_SN43916681...
corners found: 24
sn: SN43916681
mate_walking_SN43916681... OK
distance: 3.29 m


In [4]:
print(Xw)

[[ 0.19429838]
 [ 3.09417443]
 [-1.11145841]]


In [ ]:
keypoints_3d.transpose(1,0,2)[:,:,:3].shape, img.shape

### Camrea frame --> World frame

Pw = R.T @ [Pc - t]

Pc = [[((u - cx) * Zc / fx) - tx],
      [((y - cy) * Zc / fy) - ty],
      [Zc]]

In [12]:
## load stereo data
npz_load_path = "stereo_videos\\validation_test\\xyz.npz"
loaded_data = np.load(npz_load_path)

# reshape to kpt * dims * frames
keypoints_3d = loaded_data["keypoints_3d"].transpose(1,2,0)
kpt_labels = list(loaded_data["kpt_labels"])
R = loaded_data['R']
t = loaded_data['t']

print(f"loaded data : {npz_load_path}\nwith keys and shapes:")
[f"{key}:    {loaded_data[key].shape}" for key in list(loaded_data.keys())]

loaded data : stereo_videos\validation_test\xyz.npz
with keys and shapes:


['keypoints_3d:    (3908, 26, 3)',
 'kpt_labels:    (26,)',
 'R:    (3, 3)',
 't:    (3, 1)']

In [13]:
keypoints_3d_world_frame = []

for kpt in keypoints_3d:

    Zc = kpt[0,:]
    u = kpt[1,:]
    v = kpt[2,:]

    Xw = ((u-c_x) * Zc/f_x) - t[0]
    Yw = ((v-c_y) * Zc/f_y) - t[1]
    Zw = Zc - t[2]

    # multiply by -1 so height is not negative
    Yw = -Yw
    Zw = -Zw

    keypoints_3d_world_frame.append(np.array([Zw,Xw,Yw]))

    
keypoints_3d_world_frame = np.array(keypoints_3d_world_frame)
keypoints_3d_world_frame.shape 

(26, 3, 3908)

In [14]:
save_path = "./stereo_videos/validation_test/wf.npz"
with open(save_path, "w") as f:
    np.savez(
        save_path,
        keypoints_3d=keypoints_3d,
        keypoints_3d_world_frame = keypoints_3d_world_frame,
        kpt_labels=kpt_labels,
        R = R,
        t = tvec,
    )

print(f"Saved 3D keypoints to {save_path}")

Saved 3D keypoints to ./stereo_videos/validation_test/wf.npz


### test

In [33]:
print(img_points[:3])
print(obj_points[:3])

# img = cv2.imread("./stereo_videos/validation_test/mate_walking_SN43916681.png", cv2.IMREAD_GRAYSCALE)
# cv2.imshow('asdf',cv2.resize(img,(960,600)))
# cv2.waitKey(0)
# cv2.destroyAllWindows()

origin = np.array([0,0,0,1])

Rt_ = np.concat((R,tvec),axis=1)
aux = np.expand_dims(np.array([0,0,0,1]),1).T
Rt = np.concat((Rt_, aux),axis=0)
Rt_inv = np.linalg.inv(Rt)


## get pixel (u,v) from world coordinates of point
# suv = np.matrix(cameraMatrix) * np.matrix(Rt_) * np.matrix(origin).T
# suv/suv[-1]


# Rt_inv * np.matrix(origin).T
np.linalg.inv(Rt)

# print(Rt)
# print("")
# print(Rt_inv)

[[ 916.21655  990.0168 ]
 [ 976.1016   989.4064 ]
 [1036.0948   988.63556]]
[[0.   0.   0.  ]
 [0.15 0.   0.  ]
 [0.3  0.   0.  ]]


array([[ 0.9996609 , -0.01785871, -0.01895107,  0.19429838],
       [-0.01810024,  0.04665601, -0.99874701,  3.09417443],
       [ 0.01872051,  0.99875136,  0.04631695, -1.11145841],
       [ 0.        ,  0.        ,  0.        ,  1.        ]])

In [34]:
obj_points

array([[0.        , 0.        , 0.        ],
       [0.15      , 0.        , 0.        ],
       [0.3       , 0.        , 0.        ],
       [0.45000002, 0.        , 0.        ],
       [0.        , 0.15      , 0.        ],
       [0.15      , 0.15      , 0.        ],
       [0.3       , 0.15      , 0.        ],
       [0.45000002, 0.15      , 0.        ],
       [0.        , 0.3       , 0.        ],
       [0.15      , 0.3       , 0.        ],
       [0.3       , 0.3       , 0.        ],
       [0.45000002, 0.3       , 0.        ],
       [0.        , 0.45000002, 0.        ],
       [0.15      , 0.45000002, 0.        ],
       [0.3       , 0.45000002, 0.        ],
       [0.45000002, 0.45000002, 0.        ],
       [0.        , 0.6       , 0.        ],
       [0.15      , 0.6       , 0.        ],
       [0.3       , 0.6       , 0.        ],
       [0.45000002, 0.6       , 0.        ],
       [0.        , 0.75      , 0.        ],
       [0.15      , 0.75      , 0.        ],
       [0.

R @ Pw + t == [R|t]@ Pw   ???

In [ ]:
print(R)
print("\n")
print(tvec)
print("\n")
print(Rt)

Pw = np.array([1,2,3])
Pwl = np.array([1,2,3,1])

In [ ]:
first_half = np.matrix(R) * np.matrix(Pw).T + np.matrix(tvec)
second_half = np.matrix(Rt) * np.matrix(Pwl).T
print(first_half)
print('')
print(second_half)

# Step Detection

In [ ]:
# npz_load_path = "stereo_videos\\validation_test\\43916681.npz"
npz_load_path = "stereo_videos\\validation_test\\test.npz"

loaded_data = np.load(npz_load_path)

# filtered = loaded_data["filtered"]
keypoints_3d = loaded_data["keypoints_3d"]
keypoints_2d = loaded_data["keypoints_2d"]
kpt_labels = list(loaded_data["kpt_labels"])
test = loaded_data['test']

print(f"loaded data : {npz_load_path}\nwith keys and shapes:")
[f"{key}:    {loaded_data[key].shape}" for key in list(loaded_data.keys())]

In [ ]:
for ind, l in enumerate(kpt_labels):
    print(ind, l)
    

In [ ]:
gait_events, _ = step_detection(test, kpt_labels, properties)
num_ICs = len(gait_events['IC'])
num_FCs = len(gait_events['FC'])
print(f"ICs: {num_ICs}, FCs: {num_FCs}, totla: {num_FCs+num_ICs}")

In [ ]:
_

xyz stereo
